# pylevin on a GPU with jax

After the first two calls, every integral in `pylevin` is a fixed quadrature rule $I_p = \sum_n w_n f(x_n)$. This notebook exports the rules with `get_quadrature()` and evaluates repeated integrals with `jax` on the GPU. It compares them with `pylevin` itself on the CPU, and, for the examples of the `pylevin` paper, with `hankl` (correlation functions) and `CCL` (angular power spectra).

Select a GPU first: *Runtime → Change runtime type → GPU*. Installing `pylevin` compiles it, which takes a minute or two.

In [ ]:
!apt-get -qq install -y libgsl-dev libboost-dev > /dev/null
!pip -q install pylevin pyccl hankl
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import time, timeit
import numpy as np
import scipy.sparse as sparse
from scipy.interpolate import CubicSpline
import matplotlib.pyplot as plt
import jax, jax.numpy as jnp
jax.config.update("jax_enable_x64", True)
import pylevin as levin

print("jax runs on", jax.devices())

def best(function, number = 1):
    return min(timeit.repeat(function, number = number, repeat = 5))/number

def rule_matrix(lp, x, logx = True, chunk = 20000):
    """W with W[p] @ f = integral p for samples f on x (logy = False): the stored rules times the
    spline of pylevin (the not-a-knot cubic spline of scipy, zero outside x)."""
    x_nodes, weights, offsets = lp.get_quadrature()
    n_integrals = len(offsets) - 1
    integral = np.repeat(np.arange(n_integrals), np.diff(offsets))
    spline = CubicSpline(np.log(x) if logx else x, np.eye(len(x)))
    W = np.zeros((n_integrals, len(x)))
    for start in range(0, len(x_nodes), chunk):
        part = slice(start, start + chunk)
        nodes = x_nodes[part]
        values = spline(np.log(nodes) if logx else nodes)*((nodes >= x[0]) & (nodes <= x[-1]))[:, None]
        rule = sparse.csr_matrix((weights[part], (integral[part], np.arange(len(nodes)))), shape = (n_integrals, len(nodes)))
        W += rule @ values
    return W

## 1. Many integrals sharing integrands

$I(y, \ell) = \int_a^b j_\ell(xy) f(x, y)\, \mathrm{d}x$ with $f(x, y) = x^3 y + x^2 + x$ for 2000 values of $y$, each needed for 30 multipoles, so 60000 integrals. One diagonal instance with the `integrand` argument stores the 2000 integrands once. The integrands are sampled on 100 points with `logy = False`.

In [ ]:
x = np.geomspace(1e-5, 100, 100)
N = 2000
y = np.geomspace(1e-2, 1e3, N)
ells = np.arange(2, 32)
f_of_x = x[:, None]**3*y[None, :] + (x**2 + x)[:, None] # (len(x), N)

k_all = np.tile(y, len(ells))
ell_all = np.repeat(ells, N)
integrand_all = np.tile(np.arange(N), len(ells)) # variable i uses integrand integrand_all[i]
x_min, x_max = np.full(len(k_all), x[0]), np.full(len(k_all), x[-1])

lp = levin.pylevin(0, x, f_of_x, True, False, 2, True) # Colab has 2 CPU cores
lp.set_levin(10, 32, 1e-4, True, False)
result = np.zeros(len(k_all))
t0 = time.time()
for _ in range(2): # bisection, then storing the quadrature rules
    lp.levin_integrate_bessel_single(x_min, x_max, k_all, ell_all, result, integrand = integrand_all)
print(f"{len(k_all)} integrals, bisection and storing: {time.time() - t0:.1f} s")

A new integrand set costs `update_integrand` and a repeated call in `pylevin`. The sets differ by an amplitude of the $x^3 y$ term.

In [ ]:
n_sets = 50
scales = np.linspace(0.5, 1.5, n_sets)
f_sets = x[None, :, None]**3*y[None, None, :]*scales[:, None, None] + (x**2 + x)[None, :, None] # (n_sets, len(x), N)

def levin_sets():
    for f_set in f_sets:
        lp.update_integrand(x, f_set, True, False)
        lp.levin_integrate_bessel_single(x_min, x_max, k_all, ell_all, result, integrand = integrand_all)

t_levin = best(levin_sets)/n_sets
result_levin = result.copy() # the last set
print(f"pylevin: {t_levin*1e3:.3f} ms per integrand set")

### The quadrature rules in jax

The integrand is evaluated exactly at the nodes, so no interpolation is needed. `jax.vmap` integrates all sets at once, and the integrals are differentiable.

In [ ]:
x_nodes, weights, offsets = lp.get_quadrature()
n_integrals = len(offsets) - 1
integral = np.repeat(np.arange(n_integrals), np.diff(offsets)) # integral of each node
print(len(x_nodes), "nodes for", n_integrals, "integrals")

x_nodes_jax, weights_jax, integral_jax = jnp.asarray(x_nodes), jnp.asarray(weights), jnp.asarray(integral)
y_of_node = jnp.asarray(integrand_all[integral]) # integrand index of each node

@jax.jit
def integrate(y_values):
    f_nodes = x_nodes_jax**3*y_values[y_of_node] + x_nodes_jax**2 + x_nodes_jax
    return jax.ops.segment_sum(weights_jax*f_nodes, integral_jax, num_segments = n_integrals)

integrate_sets = jax.jit(jax.vmap(integrate))
y_sets = jnp.asarray(y[None, :]*scales[:, None])
integrate(y_sets[0]).block_until_ready(); integrate_sets(y_sets).block_until_ready() # compile
t_rule = best(lambda: integrate(y_sets[0]).block_until_ready(), 10)
t_rule_sets = best(lambda: integrate_sets(y_sets).block_until_ready())/n_sets

rule_sets = np.asarray(integrate_sets(y_sets))
print("differs from pylevin only by pylevin's interpolation of f:", np.max(np.abs(rule_sets[-1] - result_levin))/np.abs(result_levin).max())
gradient = jax.grad(lambda y_values: integrate(y_values).sum())(y_sets[0])

### The matrix form in jax

With `logy = False` the spline is linear in the samples, so the rules and the spline combine into a matrix `W`, and integrating sampled integrand sets is a batched matrix product: the integral for multipole `ells[l]` over integrand `j` is `W[j, l] @ f[:, j]`. This reproduces `pylevin` exactly.

In [ ]:
W = rule_matrix(lp, x).reshape(len(ells), N, len(x)).transpose(1, 0, 2).copy() # (N, len(ells), len(x))
f_stacked = np.ascontiguousarray(f_sets.transpose(2, 1, 0)) # (N, len(x), n_sets)

t_numpy = best(lambda: W @ f_stacked)/n_sets

W_64, f_64 = jnp.asarray(W), jnp.asarray(f_stacked)
W_32, f_32 = W_64.astype(jnp.float32), f_64.astype(jnp.float32)
product = jax.jit(lambda W, f: W @ f)
product(W_64, f_64).block_until_ready(); product(W_32, f_32).block_until_ready() # compile
t_matrix_64 = best(lambda: product(W_64, f_64).block_until_ready())/n_sets
t_matrix_32 = best(lambda: product(W_32, f_32).block_until_ready())/n_sets
f_one = f_stacked[..., :1]
product(W_64, jnp.asarray(f_one)).block_until_ready() # compile for one set
t_matrix_one = best(lambda: np.asarray(product(W_64, jnp.asarray(f_one))), 10) # including the transfers

matrix_64 = np.asarray(product(W_64, f_64))
matrix_32 = np.asarray(product(W_32, f_32)).astype(np.float64)
print("float64 matrix product vs pylevin:", np.max(np.abs(matrix_64[:, :, -1].T.ravel() - result_levin))/np.abs(result_levin).max())
print("float32 vs float64, relative to the largest integral of each multipole:",
      np.max(np.abs(matrix_32 - matrix_64)/np.abs(matrix_64).max(axis = (0, 2), keepdims = True)))

rows = [("pylevin, CPU (update_integrand + repeated call)", t_levin),
        ("numpy matrix product, CPU", t_numpy),
        ("jax rules, one set", t_rule),
        (f"jax rules, {n_sets} sets with vmap", t_rule_sets),
        ("jax matrix product, one set, float64", t_matrix_one),
        (f"jax matrix product, {n_sets} sets, float64", t_matrix_64),
        (f"jax matrix product, {n_sets} sets, float32", t_matrix_32)]
print(f"\n{n_integrals} integrals per integrand set, jax on {jax.devices()[0].device_kind}")
for name, t in rows:
    print(f"{name:52s} {t*1e3:9.4f} ms per set   {t_levin/t:8.1f}x")

## 2. Correlation function multipoles: comparison with hankl

The monopole of the redshift-space correlation function, $\xi_0(s) = \frac{1}{2\pi^2}\int k^2 P_0(k)\, j_0(ks)\, \mathrm{d}k$, with $P_0 = (b^2 + 2bf/3 + f^2/5)\, P_\mathrm{lin}(k)$ at six redshifts, as in the `pylevin` paper. `hankl` uses FFTLog. To avoid installing CAMB, the linear power spectrum uses the Eisenstein & Hu transfer function, normalised to $\sigma_8 = 0.81$ (close to $A_s = 2.1 \times 10^{-9}$ in the paper). A new integrand set is a new pair of bias $b$ and growth rate $f$.

In [ ]:
import pyccl as ccl
import hankl
from scipy.special import spherical_jn

cosmo = ccl.Cosmology(Omega_c = 0.27, Omega_b = 0.045, h = 0.67, sigma8 = 0.81, n_s = 0.96, transfer_function = "eisenstein_hu")
k = np.logspace(-4, 1, 2**10)
a = np.linspace(1, 0.4, 6)
pk_lin = np.array([ccl.linear_matter_power(cosmo, k, a_i) for a_i in a]) # (len(a), len(k))
s = np.linspace(2e-1, 3e2, 80)

def monopole(b, f):
    return (b*b + 2*b*f/3 + f*f/5)*pk_lin

n_pairs = 50
pairs = np.stack(np.meshgrid(np.linspace(1.5, 2.5, 10), np.linspace(0.3, 0.7, 5)), -1).reshape(-1, 2) # (b, f)

def hankl_xi(b, f):
    return [hankl.P2xi(k, p0, l = 0) for p0 in monopole(b, f)]

t_hankl = best(lambda: hankl_xi(2.0, 0.5))

lp_xi = levin.pylevin(0, k, (k**2*monopole(2.0, 0.5)).T, True, False, 2) # all s for all redshifts
lp_xi.set_levin(8, 32, 1e-4, True, False)
xi_levin = np.zeros((len(s), len(a)))
s_min, s_max = np.full(len(s), k[0]), np.full(len(s), k[-1])
for _ in range(2):
    lp_xi.levin_integrate_bessel_single(s_min, s_max, s, np.zeros(len(s), int), xi_levin)

def levin_xi(b, f):
    lp_xi.update_integrand(k, (k**2*monopole(b, f)).T, True, False)
    lp_xi.levin_integrate_bessel_single(s_min, s_max, s, np.zeros(len(s), int), xi_levin)
    return xi_levin/(2*np.pi**2)

t_levin_xi = best(lambda: levin_xi(2.0, 0.5), 5)
xi_l = levin_xi(2.0, 0.5).copy()
xi_h = hankl_xi(2.0, 0.5)

# reference at z = 0: the integral on a dense k grid
k_dense = np.geomspace(k[0], k[-1], 200001)
p0_dense = (2.0**2 + 2*2.0*0.5/3 + 0.5**2/5)*ccl.linear_matter_power(cosmo, k_dense, 1.0)
xi_brute = np.array([np.trapezoid(k_dense**2*p0_dense*spherical_jn(0, k_dense*s_i), k_dense) for s_i in s])/(2*np.pi**2)
fit = (s > 20) & (s < 250)
scale = np.max(np.abs(s[fit]**2*xi_brute[fit]))
for name, xi in (("pylevin", xi_l[:, 0]), ("hankl", np.interp(s, *xi_h[0]))):
    print(f"{name:8s} vs direct integral, max |difference in s^2 xi_0| / max |s^2 xi_0| for 20 < s < 250: {np.max(np.abs(s[fit]**2*(xi[fit] - xi_brute[fit])))/scale:.1e}")

On the GPU, all 50 pairs $(b, f)$ are one batched matrix product of `W` (per redshift: 80 separations × 1024 samples) with the samples of $k^2 P_0$.

In [ ]:
W_xi = rule_matrix(lp_xi, k).reshape(len(a), len(s), len(k))/(2*np.pi**2) # (redshift, s, k); p = redshift*len(s) + i
samples = np.stack([(k**2*monopole(b, f)) for b, f in pairs], -1) # (redshift, k, pair)
W_xi_jax, samples_jax = jnp.asarray(W_xi), jnp.asarray(samples)
xi_sets = jax.jit(lambda W, f: W @ f)
xi_sets(W_xi_jax, samples_jax).block_until_ready() # compile
t_jax_xi = best(lambda: xi_sets(W_xi_jax, samples_jax).block_until_ready())/len(pairs)
xi_sets(W_xi_jax, samples_jax[..., :1]).block_until_ready() # compile for one pair
t_jax_xi_one = best(lambda: np.asarray(xi_sets(W_xi_jax, jnp.asarray(samples[..., :1]))), 10) # including the transfers

xi_jax = np.asarray(xi_sets(W_xi_jax, samples_jax)) # (redshift, s, pair)
last = levin_xi(*pairs[-1])
print("jax matrix product vs pylevin:", np.max(np.abs(xi_jax[:, :, -1].T - last))/np.abs(last).max())
print(f"\nper (b, f) pair, six redshifts:")
for name, t in (("hankl (FFTLog), CPU", t_hankl), ("pylevin, CPU (update_integrand + repeated call)", t_levin_xi),
                (f"jax matrix product, one pair, {jax.devices()[0].platform}", t_jax_xi_one),
                (f"jax matrix product, {len(pairs)} pairs, {jax.devices()[0].platform}", t_jax_xi)):
    print(f"{name:52s} {t*1e3:9.4f} ms   {t_hankl/t:8.1f}x hankl")

fig, ax = plt.subplots()
for i in range(len(a)):
    ax.plot(xi_h[i][0], xi_h[i][0]**2*xi_h[i][1], color = "C0", lw = 2, label = "hankl" if i == 0 else None)
    ax.plot(s, s**2*xi_l[:, i], color = "C3", ls = "--", lw = 2, label = "pylevin" if i == 0 else None)
ax.set_xlim(20, s[-1]); ax.set_ylim(-100, 500)
ax.set_xlabel(r"$s\,[\mathrm{Mpc}]$"); ax.set_ylabel(r"$s^2 \xi_0$"); ax.legend()
plt.show()

## 3. Angular power spectra: comparison with CCL

The non-Limber angular power spectrum of a narrow galaxy sample at $z \approx 0.6$, as in the `pylevin` paper:

$$C_\ell = \frac{2}{\pi}\int k^2\, I_\ell(k)^2\, \mathrm{d}k\,, \qquad I_\ell(k) = \int n(\chi)\, b(\chi) \sqrt{P(k, \chi)}\; j_\ell(k\chi)\, \mathrm{d}\chi\,,$$

for 43 multipoles and 500 wavenumbers, so 21500 integrals $I_\ell(k)$. `CCL` computes the non-Limber spectra with its FFTLog-based method. One diagonal instance of `pylevin` with the `integrand` argument stores the 500 integrands, one per $k$, once. A new integrand set is a new linear bias evolution $b(z) = b_0 + b_1 (z - 0.6)$. As in the paper, the direct double integral with `scipy`'s Bessel functions serves as the reference. For this very narrow bin, CCL's non-Limber spectra did not agree with it in our tests, so compare the printed differences.

In [ ]:
cosmo = ccl.Cosmology(Omega_c = 0.25, Omega_b = 0.05, h = 0.7, sigma8 = 0.8, n_s = 0.965, transfer_function = "eisenstein_hu")
z = np.linspace(0.55, 0.65, 100)
nz = np.exp(-0.5*((z - 0.6)/0.01)**2)
ell = np.unique(np.geomspace(2, 400).astype(int))
chi = ccl.comoving_radial_distance(cosmo, 1/(1 + z))
nofchi = nz*np.gradient(z, chi)
nofchi /= np.trapezoid(nofchi, chi)
k_int = np.geomspace(1e-4, 1, 500)
sqrt_pk = np.sqrt(np.array([ccl.nonlin_matter_power(cosmo, k_int, 1/(1 + z_i)) for z_i in z])) # (len(z), len(k_int))

def ccl_cl(b0, b1):
    tracer = ccl.NumberCountsTracer(cosmo, has_rsd = False, dndz = (z, nz), bias = (z, b0 + b1*(z - 0.6)))
    return ccl.angular_cl(cosmo, tracer, tracer, ell, l_limber = 10009)

t_ccl = best(lambda: ccl_cl(1.0, 0.0))
cl_ccl = ccl_cl(1.0, 0.0)

def integrands(b0, b1):
    return sqrt_pk*(nofchi*(b0 + b1*(z - 0.6)))[:, None] # (len(chi), len(k_int))

M = len(ell)*len(k_int)
ell_all, k_all_cl = np.repeat(ell, len(k_int)), np.tile(k_int, len(ell))
integrand_cl = np.tile(np.arange(len(k_int)), len(ell))
chi_min, chi_max = np.full(M, chi[0]), np.full(M, chi[-1])
lp_cl = levin.pylevin(0, chi, integrands(1.0, 0.0), True, False, 2, True)
lp_cl.set_levin(6, 32, 5e-3, True, False)
inner = np.zeros(M)
t0 = time.time()
for _ in range(2):
    lp_cl.levin_integrate_bessel_single(chi_min, chi_max, k_all_cl, ell_all, inner, integrand = integrand_cl)
print(f"{M} integrals, bisection and storing: {time.time() - t0:.1f} s")

def levin_cl(b0, b1):
    lp_cl.update_integrand(chi, integrands(b0, b1), True, False)
    lp_cl.levin_integrate_bessel_single(chi_min, chi_max, k_all_cl, ell_all, inner, integrand = integrand_cl)
    return 2/np.pi*np.trapezoid(inner.reshape(len(ell), len(k_int))**2*k_int**2, k_int, axis = -1)

t_levin_cl = best(lambda: levin_cl(1.0, 0.0), 5)
cl_levin = levin_cl(1.0, 0.0)

# reference: the direct double integral
k_dense = np.geomspace(1e-4, 1, 2000)
sqrt_pk_dense = np.sqrt(np.array([ccl.nonlin_matter_power(cosmo, k_dense, 1/(1 + z_i)) for z_i in z]))
cl_brute = np.array([2/np.pi*np.trapezoid(np.trapezoid(nofchi[:, None]*spherical_jn(l, chi[:, None]*k_dense[None, :])*sqrt_pk_dense, chi, axis = 0)**2*k_dense**2, k_dense) for l in ell])
for name, cl in (("pylevin", cl_levin), ("CCL non-Limber", cl_ccl)):
    print(f"{name:15s} vs direct integral, maximum relative difference: {np.max(np.abs(cl/cl_brute - 1)):.1e}")

On the GPU, 50 bias evolutions are one batched matrix product of `W` (per wavenumber: 43 multipoles × 100 samples in $\chi$) with the samples, followed by the integral over $k$.

In [ ]:
W_cl = rule_matrix(lp_cl, chi).reshape(len(ell), len(k_int), len(chi)).transpose(1, 0, 2).copy() # (k, ell, chi)
biases = np.stack(np.meshgrid(np.linspace(0.8, 1.2, 10), np.linspace(-1, 1, 5)), -1).reshape(-1, 2) # (b0, b1)
samples_cl = np.stack([integrands(b0, b1).T for b0, b1 in biases], -1) # (k, chi, set)
W_cl_jax, samples_cl_jax, k_jax = jnp.asarray(W_cl), jnp.asarray(samples_cl), jnp.asarray(k_int)

@jax.jit
def cl_sets(f):
    I = W_cl_jax @ f # (k, ell, set)
    return 2/jnp.pi*jnp.trapezoid(I**2*k_jax[:, None, None]**2, k_jax, axis = 0) # (ell, set)

cl_sets(samples_cl_jax).block_until_ready() # compile
t_jax_cl = best(lambda: cl_sets(samples_cl_jax).block_until_ready())/len(biases)
cl_sets(samples_cl_jax[..., :1]).block_until_ready() # compile for one set
t_jax_cl_one = best(lambda: np.asarray(cl_sets(jnp.asarray(samples_cl[..., :1]))), 10) # including the transfers

cl_jax = np.asarray(cl_sets(samples_cl_jax))
last = levin_cl(*biases[-1])
print("jax matrix product vs pylevin:", np.max(np.abs(cl_jax[:, -1]/last - 1)))
print(f"\nper bias evolution, {len(ell)} multipoles:")
for name, t in (("CCL non-Limber (FFTLog), CPU", t_ccl), ("pylevin, CPU (update_integrand + repeated call)", t_levin_cl),
                (f"jax matrix product, one set, {jax.devices()[0].platform}", t_jax_cl_one),
                (f"jax matrix product, {len(biases)} sets, {jax.devices()[0].platform}", t_jax_cl)):
    print(f"{name:52s} {t*1e3:9.4f} ms   {t_ccl/t:8.1f}x CCL")

fig, ax = plt.subplots(2, sharex = True)
ax[0].loglog(ell, cl_brute, color = "k", lw = 2, label = "direct integral")
ax[0].loglog(ell, cl_levin, color = "C3", ls = "--", lw = 2, label = "pylevin")
ax[0].loglog(ell, cl_ccl, color = "C0", ls = ":", lw = 2, label = "CCL non-Limber")
ax[0].set_ylabel(r"$C_\ell$"); ax[0].legend()
ax[1].loglog(ell, np.abs(cl_levin/cl_brute - 1), color = "C3", label = "pylevin")
ax[1].loglog(ell, np.abs(cl_ccl/cl_brute - 1), color = "C0", ls = ":", label = "CCL non-Limber")
ax[1].set_xlabel(r"$\ell$"); ax[1].set_ylabel("relative difference"); ax[1].legend()
plt.show()

## 4. Many angular power spectra of narrow top-hat shells

Simulations such as GLASS need the angular power spectra $C_\ell^{ij}$ of the matter in many narrow shells, for all pairs of shells and every $\ell$ up to $\ell_\mathrm{max}$, and anew for every cosmology. For narrow shells the Limber approximation fails, so

$$C_\ell^{ij} = \frac{2}{\pi}\int k^2\, I_i(\ell, k)\, I_j(\ell, k)\, \mathrm{d}k\,, \qquad I_i(\ell, k) = \frac{1}{\Delta\chi_i}\int_{\chi_i}^{\chi_{i+1}} \sqrt{P(k, z(\chi))}\; j_\ell(k\chi)\, \mathrm{d}\chi$$

for top-hat shells in comoving distance. Here: 8 shells of 150 Mpc between 1000 and 2200 Mpc, every $\ell = 2, \dots, 100$ and 800 wavenumbers, so 633600 integrals $I_i(\ell, k)$ in one diagonal instance, with one integrand $\sqrt{P(k_j, z(\chi))}$ per wavenumber, sampled on 161 comoving distances, and the shell edges as integration limits. The shells are fixed in comoving distance, so that the integration limits do not change with the cosmology.

In [ ]:
from scipy.integrate import simpson

fiducial_shells = dict(Omega_c = 0.25, Omega_b = 0.05, h = 0.7, sigma8 = 0.8, n_s = 0.965)
def cosmology_shells(**parameters):
    return ccl.Cosmology(**{**fiducial_shells, **parameters}, transfer_function = "eisenstein_hu")

edges = np.linspace(1000, 2200, 9) # 8 shells of 150 Mpc
width = np.diff(edges)
chi_grid = np.linspace(edges[0], edges[-1], 161) # samples of the integrands
ells_shells = np.arange(2, 101)
k_shells = np.geomspace(1e-4, 0.3, 800)
n_shells, n_ell, n_k = len(width), len(ells_shells), len(k_shells)

def sqrt_power(cosmo):
    # one integrand sqrt(P(k_j, z(chi))) per wavenumber, shape (len(chi_grid), n_k)
    return np.sqrt(ccl.nonlin_matter_power(cosmo, k_shells, ccl.scale_factor_of_chi(cosmo, chi_grid)))

# variables (k, shell, ell), wavenumber outermost
k_var = np.repeat(k_shells, n_shells*n_ell)
shell_var = np.tile(np.repeat(np.arange(n_shells), n_ell), n_k)
ell_var = np.tile(ells_shells, n_k*n_shells)
integrand_var = np.repeat(np.arange(n_k), n_shells*n_ell) # the integrand of wavenumber k_var
chi_lo, chi_hi = edges[shell_var], edges[shell_var + 1]

w_k = np.gradient(k_shells)*k_shells**2*2/np.pi # the integral over k
def spectra_from(I):
    # C_ell^ij from I of shape (k, shell, ell), shape (ell, shell, shell), as a batched matrix product
    return I.transpose(2, 1, 0) @ (w_k[:, None, None]*I).transpose(2, 0, 1)
def spectra(result):
    return spectra_from(result.reshape(n_k, n_shells, n_ell)/width[None, :, None])
def normalised(C, C_ref):
    # max |C - C_ref| / sqrt(C_ref^ii C_ref^jj) over all ell and pairs
    return np.max(np.abs(C - C_ref)/np.sqrt(np.einsum("lii,ljj->lij", C_ref, C_ref)))

lp_shells = levin.pylevin(0, chi_grid, sqrt_power(cosmology_shells()), False, False, 2, True)
lp_shells.set_levin(6, 32, 1e-4, True, False)
result_shells = np.zeros(len(k_var))
t0 = time.time()
lp_shells.levin_integrate_bessel_single(chi_lo, chi_hi, k_var, ell_var, result_shells, integrand = integrand_var)
t_first_shells = time.time() - t0
t0 = time.time()
lp_shells.levin_integrate_bessel_single(chi_lo, chi_hi, k_var, ell_var, result_shells, integrand = integrand_var)
print(f"{len(k_var)} integrals: bisection {t_first_shells:.1f} s, storing {time.time() - t0:.1f} s")
C_fiducial = spectra(result_shells)

# reference for ell = 2, 10, 50, 100: the inner integrals with Simpson's rule on 801 points per shell
ells_check = np.array([2, 10, 50, 100])
def direct_spectra(cosmo):
    I = np.zeros((n_k, n_shells, len(ells_check)))
    for i in range(n_shells):
        chi_dense = np.linspace(edges[i], edges[i + 1], 801)
        sp = np.sqrt(ccl.nonlin_matter_power(cosmo, k_shells, ccl.scale_factor_of_chi(cosmo, chi_dense))).T # (k, chi)
        for l, ell in enumerate(ells_check):
            I[:, i, l] = simpson(sp*spherical_jn(ell, k_shells[:, None]*chi_dense[None, :]), x = chi_dense, axis = 1)/width[i]
    return spectra_from(I)

print("pylevin vs direct integral (ell = 2, 10, 50, 100):", f"{normalised(C_fiducial[ells_check - 2], direct_spectra(cosmology_shells())):.1e}")

fig, ax = plt.subplots(1, 2, figsize = (12, 4))
for i in (0, 3, 7):
    ax[0].loglog(ells_shells, C_fiducial[:, i, i], label = rf"shell {i + 1}")
    ax[1].semilogx(ells_shells, C_fiducial[:, i, min(i + 1, n_shells - 1)]/np.sqrt(C_fiducial[:, i, i]*C_fiducial[:, min(i + 1, n_shells - 1), min(i + 1, n_shells - 1)]))
ax[0].set_xlabel(r"$\ell$"); ax[0].set_ylabel(r"$C_\ell^{ii}$"); ax[0].legend()
ax[1].set_xlabel(r"$\ell$"); ax[1].set_ylabel(r"$C_\ell^{i,i+1}/\sqrt{C_\ell^{ii}C_\ell^{i+1,i+1}}$")
plt.tight_layout(); plt.show()

### Other cosmologies: updating the integrands, and jax per cosmology

As in the cosmology tutorial, for five cosmologies the spectra from `update_integrand` on the fiducial instance are compared with a new instance and with the direct integral. With `logy = False`, the rules and the spline in $\chi$ also combine into `W`, one block of (shell, $\ell$) × 161 samples per wavenumber, so `jax` computes all 633600 integrals and the spectra of a cosmology by one matrix product and one sum over $k$ on the GPU. Building `W` takes a while (it has 633600 × 161 entries), but only once.

In [ ]:
t0 = time.time()
W_shells = rule_matrix(lp_shells, chi_grid, logx = False, chunk = 200000).reshape(n_k, n_shells*n_ell, len(chi_grid))
W_shells_jax, w_k_jax, width_jax = jnp.asarray(W_shells), jnp.asarray(w_k), jnp.asarray(width)
print(f"W built in {time.time() - t0:.1f} s, {W_shells.nbytes/1e9:.2f} GB")

@jax.jit
def spectra_sets(f):
    # spectra for samples f of shape (k, chi, cosmology), shape (ell, shell, shell, cosmology)
    I = (W_shells_jax @ f).reshape(n_k, n_shells, n_ell, -1)/width_jax[None, :, None, None]
    return jnp.einsum("k,kilb,kjlb->lijb", w_k_jax, I, I)

def jax_spectra(f):
    # one cosmology, including the transfers to and from the GPU
    return np.asarray(spectra_sets(jnp.asarray(f.T[:, :, None])))[..., 0]

jax_spectra(sqrt_power(cosmology_shells())) # compile

rng_shells = np.random.default_rng(2)
draw = lambda: dict(Omega_c = rng_shells.uniform(0.22, 0.30), h = rng_shells.uniform(0.64, 0.76), sigma8 = rng_shells.uniform(0.72, 0.88), n_s = rng_shells.uniform(0.93, 1.0))

def levin_spectra(f):
    lp_shells.update_integrand(chi_grid, f, False, False)
    lp_shells.levin_integrate_bessel_single(chi_lo, chi_hi, k_var, ell_var, result_shells, integrand = integrand_var)
    return spectra(result_shells)

t_update_shells, t_jax_one_shells, t_new_shells = [], [], []
for c in range(5):
    cosmo = cosmology_shells(**draw())
    f = sqrt_power(cosmo)
    t0 = time.perf_counter(); C_updated = levin_spectra(f); t_update_shells.append(time.perf_counter() - t0)
    t0 = time.perf_counter(); C_jax_one = jax_spectra(f); t_jax_one_shells.append(time.perf_counter() - t0)
    t0 = time.perf_counter()
    lp_new = levin.pylevin(0, chi_grid, f, False, False, 2, True); lp_new.set_levin(6, 32, 1e-4, True, False)
    result_new = np.zeros(len(k_var))
    lp_new.levin_integrate_bessel_single(chi_lo, chi_hi, k_var, ell_var, result_new, integrand = integrand_var)
    t_new_shells.append(time.perf_counter() - t0)
    C_new = spectra(result_new)
    C_direct = direct_spectra(cosmo)
    print(f"cosmology {c}: vs new instance: update_integrand {normalised(C_updated, C_new):.1e}, jax {normalised(C_jax_one, C_new):.1e} | "
          f"vs direct integral: update_integrand {normalised(C_updated[ells_check - 2], C_direct):.1e}, jax {normalised(C_jax_one[ells_check - 2], C_direct):.1e}, "
          f"new instance {normalised(C_new[ells_check - 2], C_direct):.1e}")

### Many cosmologies at once on the GPU

For a batch of cosmologies, as in a sampler that evaluates many points together, the same function integrates all of them in one call.

In [ ]:
n_batch_shells = 20
f_batch_shells = np.stack([sqrt_power(cosmology_shells(**draw())).T for _ in range(n_batch_shells)], -1) # (k, chi, cosmology)
f_shells_jax = jnp.asarray(f_batch_shells)
spectra_sets(f_shells_jax).block_until_ready() # compile for the batch
t_jax_shells = best(lambda: spectra_sets(f_shells_jax).block_until_ready())/n_batch_shells
C_jax = np.asarray(spectra_sets(f_shells_jax))
print("jax vs update_integrand (last cosmology):", f"{normalised(C_jax[..., -1], levin_spectra(f_batch_shells[..., -1].T)):.1e}")

print(f"\nper cosmology, {len(k_var)} integrals and {n_ell} x {n_shells*(n_shells + 1)//2} spectra:")
for name, t in (("new instance, CPU", np.mean(t_new_shells)), ("pylevin update_integrand + repeated call, CPU", np.mean(t_update_shells)),
                (f"jax matrix product, one cosmology, {jax.devices()[0].platform}", np.mean(t_jax_one_shells)),
                (f"jax matrix product, {n_batch_shells} cosmologies, {jax.devices()[0].platform}", t_jax_shells)):
    print(f"{name:52s} {t*1e3:10.3f} ms   {np.mean(t_new_shells)/t:8.0f}x faster than a new instance")